# 04 Inference Parameter Sweep

This notebook implements the D2 grid. It is intentionally guarded by RUN_INFERENCE_SWEEP=1 because it loads SDXL and runs many GPU inferences.

In [ ]:
from pathlib import Path
import itertools
import json
import os
import yaml

from blast_pile_diffusion.data.sample_bundle import SampleBundle
from blast_pile_diffusion.inference.pipeline_builder import build_pipeline, release_pipeline
from blast_pile_diffusion.inference.controlnet_runner import run_single, save_single_result
from blast_pile_diffusion.qc.edge_alignment import check_edge_alignment

assert os.environ.get('RUN_INFERENCE_SWEEP') == '1', 'Set RUN_INFERENCE_SWEEP=1 before running this expensive notebook.'

CONFIG_PATH = Path('../configs/inference/2cn_depth_canny.yaml')
BASE_CONFIG_PATH = Path('../configs/base.yaml')
PREPROCESSED_DIR = Path('../data/preprocessed')
OUTPUT_DIR = Path('../data/generated/param_sweep')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bundle_dirs = sorted(p for p in PREPROCESSED_DIR.iterdir() if (p / 'rgb.png').exists())[:5]
assert 3 <= len(bundle_dirs) <= 5, 'D2 expects 3-5 representative bundles.'
bundles = [SampleBundle.load(path) for path in bundle_dirs]
pipe, base_cfg = build_pipeline(CONFIG_PATH, BASE_CONFIG_PATH)


In [ ]:
depth_scales = [0.5, 0.7, 0.9]
canny_scales = [0.4, 0.6, 0.8]
strengths = [0.30, 0.40, 0.50, 0.60]
guidances = [5.0, 6.5, 8.0]
results = []

try:
    for depth_scale, canny_scale, strength, guidance in itertools.product(depth_scales, canny_scales, strengths, guidances):
        cfg = yaml.safe_load(json.dumps(base_cfg))
        for cn in cfg['controlnets']:
            if cn['name'] == 'depth':
                cn['scale'] = depth_scale
            if cn['name'] == 'canny':
                cn['scale'] = canny_scale
        cfg['inference']['strength'] = strength
        cfg['inference']['guidance_scale'] = guidance
        offsets = []
        for bundle in bundles:
            generated = run_single(pipe, bundle, cfg, seed=42)
            result = check_edge_alignment(generated, bundle.mask)
            offsets.append(result.mean_offset)
            result_dir = OUTPUT_DIR / f"{bundle.sample_key}_d{depth_scale}_c{canny_scale}_s{strength}_g{guidance}"
            save_single_result(result_dir, bundle, generated, cfg, seed=42)
        results.append({
            'depth_scale': depth_scale,
            'canny_scale': canny_scale,
            'strength': strength,
            'guidance_scale': guidance,
            'mean_offset': float(sum(offsets) / len(offsets)),
        })
finally:
    release_pipeline(pipe)

results = sorted(results, key=lambda item: item['mean_offset'])
(OUTPUT_DIR / 'param_sweep_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
results[:10]


In [ ]:
best = results[0]
print('Best parameters:', best)
print('Review thumbnails under', OUTPUT_DIR, 'then write approved values back to configs/inference/2cn_depth_canny.yaml.')
